# Member 1: Evaluation Metrics & Model Comparison

This notebook completes the Member 1 part of Phase 4, **Evaluation & Discussion**. It uses the saved test-set predictions from the Model Development phase and calculates the required regression metrics for each model.

Required outputs:
- Evaluation code
- Model comparison table
- Best model result
- Short explanation of metrics

## 1. Load Test Predictions

The model training has already been completed in `Model_Development/AAPL_Model_Development.ipynb`. This notebook starts from the saved test predictions so the evaluation results are reproducible and consistent with the model-development notebook.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_DIR = Path.cwd().parents[1]
RESULTS_DIR = PROJECT_DIR / "Model_Development" / "results"
MODELS_DIR = PROJECT_DIR / "Model_Development" / "models"
OUTPUT_DIR = Path.cwd()

prediction_path = RESULTS_DIR / "test_predictions.csv"
metadata_path = MODELS_DIR / "best_model_metadata.json"

test_predictions = pd.read_csv(prediction_path, parse_dates=["Date"])
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print(f"Loaded test predictions: {test_predictions.shape[0]} rows")
print(f"Test period: {test_predictions['Date'].min().date()} to {test_predictions['Date'].max().date()}")
display(test_predictions.head())

Loaded test predictions: 251 rows
Test period: 2024-01-02 to 2024-12-30


,Date,Current_Close,Actual_Target_Close,Persistence_Baseline,Linear Regression,Ridge Regression,Random Forest,XGBoost
0,2024-01-02,183.562195,182.187744,183.562195,183.824285,183.814878,184.337339,179.62534
1,2024-01-03,182.187744,179.873962,182.187744,183.084667,183.058693,183.954662,181.80950
2,2024-01-04,179.873962,179.152084,179.873962,180.456630,180.433810,179.743782,179.21786
3,2024-01-05,179.152084,183.483063,179.152084,179.381588,179.390411,178.489362,176.63324
4,2024-01-08,183.483063,183.067780,183.483063,183.404450,183.371389,183.308599,180.72844


## 2. Confirm Evaluated Models

The required models are evaluated below:

1. Persistence Baseline
2. Linear Regression
3. Ridge Regression
4. Random Forest
5. XGBoost

In [2]:
model_columns = [
    "Persistence_Baseline",
    "Linear Regression",
    "Ridge Regression",
    "Random Forest",
    "XGBoost",
]

missing_models = [model for model in model_columns if model not in test_predictions.columns]
if missing_models:
    raise ValueError(f"Missing prediction columns: {missing_models}")

print("All required model prediction columns are available.")

All required model prediction columns are available.


## 3. Calculate Regression Metrics

Metrics used:

- **MAE**: average absolute prediction error.
- **MSE**: average squared prediction error.
- **RMSE**: square root of MSE; this is the main model-selection metric because it penalizes large errors more strongly.
- **R2 Score**: proportion of price-level variation explained by the model.
- **Directional Accuracy**: percentage of days where the predicted price movement direction matches the actual next-day movement direction.

In [3]:
def directional_accuracy(current_close, actual_next_close, predicted_next_close):
    actual_direction = np.sign(actual_next_close - current_close)
    predicted_direction = np.sign(predicted_next_close - current_close)
    return float(np.mean(actual_direction == predicted_direction))

y_true = test_predictions["Actual_Target_Close"]
current_close = test_predictions["Current_Close"]

rows = []
for model in model_columns:
    y_pred = test_predictions[model]
    mse = mean_squared_error(y_true, y_pred)
    rows.append({
        "Model": model.replace("_", " "),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2 Score": r2_score(y_true, y_pred),
        "Directional Accuracy": directional_accuracy(current_close, y_true, y_pred),
    })

comparison_table = pd.DataFrame(rows)
comparison_table["Locked Selection"] = comparison_table["Model"].eq(metadata["locked_model_name"])
comparison_table = comparison_table.sort_values("RMSE", ascending=True).reset_index(drop=True)

full_precision_path = OUTPUT_DIR / "member1_model_comparison_full_precision.csv"
presentation_path = OUTPUT_DIR / "member1_model_comparison_table.csv"
comparison_table.to_csv(full_precision_path, index=False, encoding="utf-8-sig")

presentation_table = comparison_table.copy()
for col in ["MAE", "MSE", "RMSE", "R2 Score", "Directional Accuracy"]:
    presentation_table[col] = presentation_table[col].round(4)
presentation_table.to_csv(presentation_path, index=False, encoding="utf-8-sig")

display(presentation_table)
print(f"Saved: {presentation_path.name}")
print(f"Saved: {full_precision_path.name}")

,Model,MAE,MSE,RMSE,R2 Score,Directional Accuracy,Locked Selection
0,Persistence Baseline,2.1047,8.1830,2.8606,0.9874,0.0040,False
1,Ridge Regression,2.1196,8.2735,2.8764,0.9873,0.4900,True
2,Linear Regression,2.1212,8.2926,2.8797,0.9873,0.5060,False
3,Random Forest,19.4153,687.6152,26.2224,-0.0563,0.4024,False
4,XGBoost,19.8951,721.6218,26.8630,-0.1085,0.4502,False


Saved: member1_model_comparison_table.csv
Saved: member1_model_comparison_full_precision.csv


## 4. Best Model Result

The best machine learning model is selected using validation RMSE during Model Development, then reported on the unseen 2024 test set.

In [4]:
best_model_name = metadata["locked_model_name"]
best_row = comparison_table.loc[comparison_table["Model"].eq(best_model_name)].iloc[0]

best_summary = pd.DataFrame([{
    "Best ML Model": best_model_name,
    "Test MAE": best_row["MAE"],
    "Test MSE": best_row["MSE"],
    "Test RMSE": best_row["RMSE"],
    "Test R2 Score": best_row["R2 Score"],
    "Test Directional Accuracy": best_row["Directional Accuracy"],
}])

best_summary_rounded = best_summary.copy()
for col in best_summary_rounded.columns[1:]:
    best_summary_rounded[col] = best_summary_rounded[col].round(4)

best_summary_rounded.to_csv(OUTPUT_DIR / "member1_best_model_result.csv", index=False, encoding="utf-8-sig")
display(best_summary_rounded)

,Best ML Model,Test MAE,Test MSE,Test RMSE,Test R2 Score,Test Directional Accuracy
0,Ridge Regression,2.1196,8.2735,2.8764,0.9873,0.49


## 5. Member 1 Explanation

Based on the evaluation results, **Ridge Regression** was selected as the best machine learning model because it achieved the lowest validation RMSE among the trained ML models and remained stable on the unseen 2024 test set. RMSE was used as the main metric because it gives a higher penalty to large prediction errors, which is important in stock price prediction.

On the test set, Ridge Regression achieved an RMSE of about **2.8764** and an R2 Score of about **0.9873**. This means the model followed the overall AAPL price level well. However, the Directional Accuracy was only about **49.00%**, so the model should not be interpreted as a reliable trading signal. It is better understood as an academic price-level prediction model.

In [5]:
explanation = f"""# Member 1 Evaluation Metrics Explanation

Based on the evaluation results, **{best_model_name}** was selected as the best machine learning model because it achieved the lowest validation RMSE among the trained ML models and remained stable on the unseen 2024 test set. RMSE was used as the main metric because it gives a higher penalty to large prediction errors, which is important in stock price prediction.

On the test set, {best_model_name} achieved:

- MAE: {best_row['MAE']:.4f}
- MSE: {best_row['MSE']:.4f}
- RMSE: {best_row['RMSE']:.4f}
- R2 Score: {best_row['R2 Score']:.4f}
- Directional Accuracy: {best_row['Directional Accuracy']:.4f}

The high R2 Score shows that the model followed the overall AAPL price level well. However, Directional Accuracy was not high, so the model should not be used as a reliable trading signal or financial advice. It is mainly suitable for academic learning and price-level prediction.
"""

(OUTPUT_DIR / "member1_metrics_explanation.md").write_text(explanation, encoding="utf-8-sig")
display(Markdown(explanation))
print("Saved: member1_metrics_explanation.md")

# Member 1 Evaluation Metrics Explanation

Based on the evaluation results, **Ridge Regression** was selected as the best machine learning model because it achieved the lowest validation RMSE among the trained ML models and remained stable on the unseen 2024 test set. RMSE was used as the main metric because it gives a higher penalty to large prediction errors, which is important in stock price prediction.

On the test set, Ridge Regression achieved:

- MAE: 2.1196
- MSE: 8.2735
- RMSE: 2.8764
- R2 Score: 0.9873
- Directional Accuracy: 0.4900

The high R2 Score shows that the model followed the overall AAPL price level well. However, Directional Accuracy was not high, so the model should not be used as a reliable trading signal or financial advice. It is mainly suitable for academic learning and price-level prediction.


Saved: member1_metrics_explanation.md


## 6. Member 1 Deliverables

This folder contains the files that Member 1 can send to the group leader:

- `Member1_Evaluation_Metrics.ipynb`
- `member1_model_comparison_table.csv`
- `member1_model_comparison_full_precision.csv`
- `member1_best_model_result.csv`
- `member1_metrics_explanation.md`